# 5. Feature Engineering & Persiapan Supervised Learning

Tahap ini mengubah `main_trips` jadi dataset siap model:

- agregasi pickup jadi **zona × jam** (grid lengkap, isi 0 kalau tidak ada pickup)
- fitur waktu: `hour_of_day`, `day_of_week`, `month`, `is_weekend`, `is_holiday`, `is_major_nyc_event`
- fitur historis per zona: `lag_1`, `lag_24`, `lag_168`, `roll_24`, `roll_168` (sudah di-shift)
- fitur lokasi: koordinat pickup + `borough`, `is_airport`, `is_cbd`
- target regresi: `pickup_count` (jam t+1), target klasifikasi: `demand_level`
- split kronologis: Train Jan-Apr, Validation Mei, Test Juni
- deret waktu kota & Top-5 zona untuk analisis time series

Aturan anti-bocor: semua fitur historis pakai data **sebelum** jam yang diprediksi, tidak ada koordinat dropoff atau nilai transaksi.

# 0. Import Library

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Agregasi Demand Zona-Jam

In [ ]:
# cuma butuh waktu dan koordinat pickup buat agregasi
trips = pd.read_parquet("../datasets/interim/2026/main_trips.parquet",
                        columns=["tpep_pickup_datetime", "PU_longitude", "PU_latitude"])
trips["hour"] = trips["tpep_pickup_datetime"].dt.floor("h")

# kunci zona = pasangan koordinat, dibulatkan 6 desimal biar aman saat merge
trips["lo6"] = trips["PU_longitude"].round(6)
trips["la6"] = trips["PU_latitude"].round(6)

count = (trips.groupby(["lo6", "la6", "hour"]).size()
         .rename("pickup_count").reset_index())
print("Kombinasi zona-jam berisi pickup:", len(count))
print("Total pickup:", int(count["pickup_count"].sum()))
del trips

# 2. Grid Zona-Jam Lengkap

In [ ]:
zona = pd.read_csv("../datasets/zone/zone_centroid_lookup.csv")
zona = zona.rename(columns={"LocationID": "zone_id", "longitude": "PU_longitude", "latitude": "PU_latitude",
                            "zone_name": "PU_zone_name", "borough": "PU_borough"})
zona["lo6"] = zona["PU_longitude"].round(6)
zona["la6"] = zona["PU_latitude"].round(6)
assert zona[["lo6", "la6"]].drop_duplicates().shape[0] == len(zona), "koordinat zona tidak unik"
print("Jumlah zona:", len(zona))

jam = pd.date_range("2026-01-01 00:00", "2026-06-30 23:00", freq="h")
print("Jumlah jam:", len(jam))

grid = zona.merge(pd.DataFrame({"hour": jam}), how="cross")
grid = grid.merge(count, on=["lo6", "la6", "hour"], how="left")
grid["pickup_count"] = grid["pickup_count"].fillna(0)  # zona-jam tanpa pickup = 0

# 8 Maret 2026 jam 02:00 tidak ada (DST), tandai tidak tersedia bukan 0
dst = pd.Timestamp("2026-03-08 02:00")
grid["is_dst_missing"] = (grid["hour"] == dst).astype(int)
grid.loc[grid["is_dst_missing"] == 1, "pickup_count"] = np.nan

grid = grid.sort_values(["zone_id", "hour"]).reset_index(drop=True)
print("Grid zona-jam:", len(grid))
print("Total pickup di grid (harus = total di atas):", int(np.nansum(grid["pickup_count"])))

# 3. Fitur Waktu, Libur, dan Lokasi

In [ ]:
grid["hour_of_day"] = grid["hour"].dt.hour
grid["day_of_week"] = grid["hour"].dt.dayofweek   # 0 = Senin
grid["month"] = grid["hour"].dt.month
grid["is_weekend"] = (grid["day_of_week"] >= 5).astype(int)

# kalender libur/event resmi (dari Anggota 4)
kalender = pd.read_csv("../datasets/processed/nyc_holiday_event_calendar_2026.csv", parse_dates=["date"])
kalender = kalender.set_index(kalender["date"].dt.normalize())
tgl = grid["hour"].dt.normalize()
grid["is_holiday"] = tgl.map(kalender["is_holiday"]).fillna(0).astype(int)
grid["is_major_nyc_event"] = tgl.map(kalender["is_major_nyc_event"]).fillna(0).astype(int)

print("Tanggal federal holiday:", int(kalender["is_holiday"].sum()))
print("Tanggal major NYC event:", int(kalender["is_major_nyc_event"].sum()))
print(grid[["hour_of_day", "day_of_week", "month", "is_weekend", "is_holiday", "is_major_nyc_event"]].describe().T[["min", "max"]])

# 4. Fitur Historis per Zona (lag & rolling, sudah di-shift)

In [ ]:
g = grid.groupby("zone_id", sort=False)
grid["lag_1"] = g["pickup_count"].shift(1)
grid["lag_24"] = g["pickup_count"].shift(24)
grid["lag_168"] = g["pickup_count"].shift(168)

# rolling mean pakai data sebelum jam ini (shift dulu, baru rolling)
grid["roll_24"] = g["pickup_count"].transform(lambda s: s.shift(1).rolling(24, min_periods=24).mean())
grid["roll_168"] = g["pickup_count"].transform(lambda s: s.shift(1).rolling(168, min_periods=168).mean())

print("NaN tiap fitur historis (awal seri per zona):")
print(grid[["lag_1", "lag_24", "lag_168", "roll_24", "roll_168"]].isna().sum())

# 5. Target, Split Kronologis, dan demand_level

In [ ]:
# target = pickup_count pada jam ini (fitur dihitung dari jam-jam sebelumnya)
grid["target_pickup_count"] = grid["pickup_count"]

# split kronologis
grid["split"] = np.select(
    [grid["month"] <= 4, grid["month"] == 5],
    ["train", "validation"], default="test")
print(grid["split"].value_counts())

# 6. Dataset Modeling

Baris dengan fitur historis belum lengkap (awal seri tiap zona) dibuang. Batas `demand_level` dihitung **hanya dari data training**.

In [ ]:
FITUR = ["hour_of_day", "day_of_week", "month", "is_weekend", "is_holiday", "is_major_nyc_event",
         "PU_longitude", "PU_latitude", "is_airport", "is_cbd",
         "lag_1", "lag_24", "lag_168", "roll_24", "roll_168"]
TARGET = "target_pickup_count"

dataset = grid.dropna(subset=FITUR + [TARGET]).reset_index(drop=True)
dataset["demand_level"] = None

# batas kelas dari training saja (tercile 33/66)
q1, q2 = dataset.loc[dataset["split"] == "train", TARGET].quantile([1 / 3, 2 / 3])
if q1 == q2:
    q2 = q1 + 1
dataset["demand_level"] = pd.cut(dataset[TARGET], [-np.inf, q1, q2, np.inf], labels=["Rendah", "Sedang", "Tinggi"])

print(f"Batas kelas: Rendah <= {q1:.2f}, Sedang <= {q2:.2f}, Tinggi > {q2:.2f}")
print("Baris modeling:", len(dataset))
print(dataset["split"].value_counts())
print("Distribusi demand_level (train):")
print(dataset.loc[dataset["split"] == "train", "demand_level"].value_counts(normalize=True).round(3))

# 7. Pemeriksaan Leakage

In [ ]:
dilarang = ["DO_longitude", "DO_latitude", "fare_amount", "total_amount",
            "tip_amount", "cbd_congestion_fee", "trip_distance", "passenger_count"]
bocor = [c for c in dilarang if c in FITUR]
assert not bocor, f"Fitur bocor: {bocor}"
assert TARGET not in FITUR, "target masuk fitur!"

# cek lag_1 memang pickup jam sebelumnya (sample zona pertama)
s = grid[grid["zone_id"] == grid["zone_id"].iloc[0]].sort_values("hour")
cek = np.allclose(s["lag_1"], s["pickup_count"].shift(1), equal_nan=True)
print("lag_1 = pickup jam sebelumnya:", bool(cek))
print("Fitur aman:", FITUR)

# 8. Simpan Dataset & Deret Waktu

In [ ]:
os.makedirs("../datasets/interim/2026", exist_ok=True)

# grid zona-jam (buat EDA, agregasi, spasial)
grid_out = grid[["hour", "zone_id", "PU_longitude", "PU_latitude", "PU_zone_name", "PU_borough",
                 "is_airport", "is_cbd", "pickup_count", "is_dst_missing"]]
grid_out.to_parquet("../datasets/interim/2026/demand_zone_hour.parquet", index=False)

# dataset fitur + target (kolom internal dibuang biar tidak salah dipakai)
simpan = ["hour", "zone_id", "PU_longitude", "PU_latitude", "PU_zone_name", "PU_borough",
          "is_airport", "is_cbd", "hour_of_day", "day_of_week", "month", "is_weekend", "is_holiday", "is_major_nyc_event",
          "lag_1", "lag_24", "lag_168", "roll_24", "roll_168",
          "target_pickup_count", "demand_level", "split"]
dataset[simpan].to_parquet("../datasets/interim/2026/features_dataset.parquet", index=False)

# deret waktu kota keseluruhan
kota = grid.groupby("hour")["pickup_count"].sum(min_count=1).reset_index().rename(columns={"pickup_count": "total_pickup"})
kota.to_parquet("../datasets/interim/2026/timeseries_city.parquet", index=False)

# deret waktu Top-5 zona
total_zona = grid.groupby(["zone_id", "PU_zone_name"])["pickup_count"].sum(min_count=1).reset_index()
top5 = total_zona.nlargest(5, "pickup_count")
ts_top5 = grid[grid["zone_id"].isin(top5["zone_id"])][["hour", "zone_id", "PU_zone_name", "pickup_count"]].sort_values(["zone_id", "hour"])
ts_top5.to_parquet("../datasets/interim/2026/timeseries_top5.parquet", index=False)

print("Top-5 zona:")
print(top5.to_string(index=False))
print("\ndemand_zone_hour:", grid_out.shape)
print("features_dataset:", dataset.shape)
print("timeseries_city:", kota.shape)
print("timeseries_top5:", ts_top5.shape)

# 9. Ringkasan

In [ ]:
print("=== Ringkasan Feature Engineering ===")
print(f"Baris dataset fitur : {len(dataset):,}")
print(f"Jumlah fitur        : {len(FITUR)}")
print(f"Periode train       : {dataset.loc[dataset.split=='train','hour'].min()} s/d {dataset.loc[dataset.split=='train','hour'].max()}")
print(f"Periode validation  : {dataset.loc[dataset.split=='validation','hour'].min()} s/d {dataset.loc[dataset.split=='validation','hour'].max()}")
print(f"Periode test        : {dataset.loc[dataset.split=='test','hour'].min()} s/d {dataset.loc[dataset.split=='test','hour'].max()}")
print()
print(dataset[FITUR + [TARGET]].describe().T[["mean", "std", "min", "max"]].round(2))

## Catatan

Daftar fitur lengkap (nama, tipe, definisi, tersedia saat prediksi atau tidak) ada di `docs/feature_dictionary.md`.